# Random ID Baseline — Thesis Null Hypothesis (Colab)

**目的**：把 RQ-VAE Semantic ID 替换成均匀随机采样的 4-token ID，其他全部保持不变，跑同一个 T5。如果指标暴跌，证明 +25% N@10 的功劳来自 **语义结构**，而不是 T5 架构 / 数据增强 / 评估协议。

**运行前**：Runtime → Change runtime type → **GPU**（强烈推荐 A100；T4 需要手动降 `batch_size`，见下方 warning）

**无需上传任何文件** —— `data/beauty_data.pkl` 已在 git 里，随 clone 自带；随机 ID 在 Colab 上现场生成。

---

## 运行顺序
1. **环境**（Cell 1–3）：GPU 检查 → 装依赖 → clone repo
2. **生成随机 ID**（Cell 4）：`generate_random_ids.py`
3. **训练 + 评估**（Cell 5–6）：T5 on random SIDs
4. **下载结果**（Cell 7，可选）

In [ ]:
# Cell 1：GPU 检查 + batch size 建议
import torch
print(f'GPU 可用: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU 型号: {name}')
    print(f'显存:     {mem:.1f} GB')
    if 'A100' in name:
        print('\n✅ A100 检测到，保持默认 batch_size=512，与主线 RQ-VAE run 硬件一致。')
    elif 'T4' in name or mem < 20:
        print('\n⚠️  非 A100 GPU 显存偏小。默认 batch_size=512 可能 OOM。')
        print('    如果训练启动后显存爆了，打开 train.py，把 CONFIG["batch_size"] 改成 256。')
        print('    注意：这会引入一个 batch-size 变量，严格讲与主线不完全对齐，')
        print('    但 Adam 对 batch size 相对鲁棒，随机 ID 这种预期大幅劣化的 baseline 可以接受。')
    else:
        print(f'\n  {name} 未识别，保持默认试试，OOM 再降 batch_size。')
else:
    print('\n❌ 未检测到 GPU。Runtime → Change runtime type → GPU。')

In [ ]:
# Cell 2：安装依赖
!pip install transformers scikit-learn -q
print('依赖安装完成 ✅')

In [ ]:
# Cell 3：clone 仓库（fresh session）或 pull（复用 session）
import os
REPO = 'Generative-Sequential-Recommendation'
if not os.path.exists(REPO):
    !git clone https://github.com/rhine-yangrui/Generative-Sequential-Recommendation.git
else:
    print('repo 已存在，执行 git pull 同步最新提交')
    !cd {REPO} && git pull
%cd {REPO}
!git log --oneline -3
print()
!ls data/ embedding/

In [ ]:
# Cell 4：生成随机 4-token IDs（seed=42，item 间唯一）
!python embedding/generate_random_ids.py

# Sanity check：加载回来看形状和分布
import numpy as np
sids = np.load('embedding/semantic_ids_random.npy', allow_pickle=True).item()
vals = list(sids.values())
print(f'\n✅ 加载成功：{len(sids)} items')
print(f'   示例：item {list(sids.keys())[0]} → {vals[0]}')
assert len(vals[0]) == 4, f'期望 4-token ID，实际 {len(vals[0])}'

ranges = list(zip(*vals))
for i, r in enumerate(ranges):
    print(f'   c{i} 范围: {min(r)}..{max(r)}  usage: {len(set(r))}')

unique_ids = len({tuple(v) for v in vals})
assert unique_ids == len(sids), f'仍有冲突: {unique_ids} vs {len(sids)}'
print(f'   唯一 4-tuple: {unique_ids} / {len(sids)}  ✓')

---
## 训练 + 评估

预期：random ID 上 val NDCG@10 学不起来，patience=10 / val_every=2 会在 **~20 epoch** 触发早停，A100 实际耗时 **15–25 分钟**（远小于主线 ~130 epoch）。

判读准则：

| Test R@10 | 含义 | 行动 |
|---|---|---|
| < 0.01 | Thesis 强成立 | 更新 Progress.md，推进冷启动分片 |
| 0.01 – 0.03 | Thesis 部分成立 | 正常记录，分析 gap |
| > 0.03 | ⚠️ 需要排查 data leakage | 先停，别急着下结论 |
| ≥ 0.058 (≈ RQ-VAE) | Thesis 被推翻 | 立即重新设计实验 |

In [ ]:
# Cell 5：训练 T5 on random SIDs
# 与主线唯一区别：--semantic-ids 指向 random，--ckpt 另存避免覆盖主线
!python train.py \
    --semantic-ids semantic_ids_random.npy \
    --ckpt checkpoints/best_model_t5_random.pt

In [ ]:
# Cell 6：评估（all-rank Recall@K / NDCG@K，beam=50）
!python evaluate.py \
    --semantic-ids semantic_ids_random.npy \
    --ckpt checkpoints/best_model_t5_random.pt

---
## 下载结果（可选）

如果想保留 random ID 的 checkpoint 和产物以便复现，取消注释 Cell 7 运行。注意：random ID checkpoint 对后续实验没用，不下载也没关系 —— 记住数字就够了。

In [ ]:
# Cell 7：下载 checkpoint + random SIDs（可选）
# from google.colab import files
# import os
# for f in ['checkpoints/best_model_t5_random.pt', 'embedding/semantic_ids_random.npy']:
#     if os.path.exists(f):
#         files.download(f)
#         print(f'✅ {f}  ({os.path.getsize(f)/1e6:.1f} MB)')
#     else:
#         print(f'⚠️  {f} 不存在')